In [1]:
import requests
try:
    requests.get("http://localhost:11434", timeout=2)
    print("Ollama is reachable")
except requests.exceptions.ConnectionError:
    raise RuntimeError("Ollama isn't running. Start it with `ollama serve` or open the Ollama app.")

Ollama is reachable


In [13]:
import os, psycopg, sys, numpy as np
from app.rag import get_embedder

sys.path.append(r"C:\Git projects\sales_marketing_rag")
                
embed = get_embedder()

conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

from pgvector.psycopg import register_vector
register_vector(conn)

rows = conn.execute("SELECT id, content FROM chunks WHERE embedding IS NULL ORDER BY id").fetchall()
BATCH = 128
for i in range(0, len(rows), BATCH):
    batch = rows[i:i+BATCH]
    vecs = embed([c for _, c in batch])
    with conn.cursor() as cur:
        cur.executemany("UPDATE chunks SET embedding = %s WHERE id = %s",
                        [(np.array(v), cid) for (cid, _), v in zip(batch, vecs)])
    print(f"embedded {i+len(batch)}/{len(rows)}")

embedded 73/73


In [14]:
conn.execute("CREATE INDEX IF NOT EXISTS chunks_hnsw "
             "ON chunks USING hnsw (embedding vector_cosine_ops);")   # vector side
conn.execute("CREATE INDEX IF NOT EXISTS chunks_tsv "
             "ON chunks USING GIN (tsv);")                            # keyword side


<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost port=5433 database=rag) at 0x1c2f3299e40>